In [ ]:
# ======================================================================================
# BROWSER-BASED ECG ANCHOR ANNOTATION TOOL
#
# NO matplotlib interactivity
# NO ipympl
# NO Jupyter widgets
#
# Creates:
#     ecg_anchor_annotator.html
#
# Workflow:
#   1. Run this cell.
#   2. Open ecg_anchor_annotator.html in Chrome/Safari.
#   3. For each displayed ECG lead, click along the TRUE black ECG trace
#      from LEFT -> RIGHT.
#   4. Use Undo/Clear if needed.
#   5. Mark "Keep V4" if the existing automatic trace is acceptable.
#   6. Click Export JSON when finished.
#   7. Move downloaded ecg_anchor_annotations.json into your project folder.
#
# The page automatically saves progress in browser localStorage.
# ======================================================================================

import os
import json
import base64
import webbrowser

from pathlib import Path
from io import BytesIO
from PIL import Image


# ======================================================================================
# 1. PATHS
# ======================================================================================

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)

if not GEOMETRY_PATH.exists():
    matches = list(
        Path(".").glob(
            "**/manual_geometry.json"
        )
    )

    if not matches:
        raise FileNotFoundError(
            "Could not find manual_geometry.json"
        )

    GEOMETRY_PATH = matches[0]

print("Using geometry:", GEOMETRY_PATH)

with open(
    GEOMETRY_PATH,
    "r"
) as f:
    geometry = json.load(f)


OUTPUT_HTML = Path(
    "ecg_anchor_annotator.html"
)


# ======================================================================================
# 2. LEADS WE WANT TO REVIEW/CORRECT
# ======================================================================================

CORRECT_LEADS = {

    "CompleteHeartBlock": [
        "V1",
        "V2",
        "V3",
    ],

    "Hyperkalemia": [
        "aVL",
        "V1",
        "V2",
        "V3",
        "V4",
    ],

    "LBBB": [
        "aVL",
        "V1",
        "V2",
        "V3",
    ],

    "STEMI": [
        "III",
        "aVR",
        "aVL",
        "aVF",
        "V1",
        "V2",
        "V4",
        "V5",
        "V6",
    ],

    "VentPaced": [
        "V2",
        "V3",
        "V4",
        "V5",
    ],

    "VTach": [
        "II",
        "III",
        "aVL",
        "aVF",
        "V2",
    ],

    "WPW": [
        "V1",
    ],
}


DISPLAY_GRID = [
    ["I", "aVR", "V1", "V4"],
    ["II", "aVL", "V2", "V5"],
    ["III", "aVF", "V3", "V6"],
]


DISPLAY_POSITION = {}

for row_i, row in enumerate(
    DISPLAY_GRID
):
    for col_i, lead in enumerate(
        row
    ):
        DISPLAY_POSITION[lead] = (
            row_i,
            col_i
        )


# ======================================================================================
# 3. SAME V4 CROP GEOMETRY WE ALREADY ACCEPTED
# ======================================================================================

ROW_HALF_HEIGHT_SCALE = 0.62
X_PAD_FRACTION = 0.025


def resolve_image_path(path):

    p = Path(path)

    if p.exists():
        return p

    basename = p.name

    matches = list(
        Path(".").glob(
            f"**/{basename}"
        )
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not find image {basename}"
        )

    return matches[0]


def get_lead_box(
    ecg_key,
    lead
):

    g = geometry[ecg_key]

    rows = g["row_centres"]
    xs = g["x_boundaries"]

    H = int(
        g["image_height"]
    )

    W = int(
        g["image_width"]
    )

    row_i, col_i = (
        DISPLAY_POSITION[lead]
    )

    if row_i == 0:
        spacing = (
            rows[1]
            -
            rows[0]
        )

    elif row_i == 2:
        spacing = (
            rows[2]
            -
            rows[1]
        )

    else:
        spacing = min(
            rows[1] - rows[0],
            rows[2] - rows[1]
        )

    half_height = (
        ROW_HALF_HEIGHT_SCALE
        *
        spacing
    )

    centre_y = rows[
        row_i
    ]

    y0 = max(
        0,
        int(
            round(
                centre_y
                -
                half_height
            )
        )
    )

    y1 = min(
        H,
        int(
            round(
                centre_y
                +
                half_height
            )
        )
    )

    x0 = xs[
        col_i
    ]

    x1 = xs[
        col_i + 1
    ]

    width = (
        x1 - x0
    )

    x0 = int(
        round(
            x0
            +
            X_PAD_FRACTION
            *
            width
        )
    )

    x1 = int(
        round(
            x1
            -
            X_PAD_FRACTION
            *
            width
        )
    )

    return (
        x0,
        y0,
        x1,
        y1,
    )


# ======================================================================================
# 4. EMBED CROPS AS BASE64 PNG
# ======================================================================================

def pil_to_data_uri(
    image
):

    buffer = BytesIO()

    image.save(
        buffer,
        format="PNG"
    )

    encoded = (
        base64.b64encode(
            buffer.getvalue()
        )
        .decode(
            "ascii"
        )
    )

    return (
        "data:image/png;base64,"
        +
        encoded
    )


tasks = []


for ecg_key, leads in (
    CORRECT_LEADS.items()
):

    source_path = (
        resolve_image_path(
            geometry[
                ecg_key
            ][
                "image_path"
            ]
        )
    )

    source = (
        Image.open(
            source_path
        )
        .convert(
            "RGB"
        )
    )

    for lead in leads:

        x0, y0, x1, y1 = (
            get_lead_box(
                ecg_key,
                lead
            )
        )

        crop = source.crop(
            (
                x0,
                y0,
                x1,
                y1
            )
        )

        # Upscale somewhat so clicking is easier
        max_width = 1400

        if crop.width < max_width:

            scale = (
                max_width
                /
                crop.width
            )

            crop = crop.resize(
                (
                    int(
                        crop.width
                        *
                        scale
                    ),
                    int(
                        crop.height
                        *
                        scale
                    ),
                ),
                Image.Resampling.LANCZOS
            )

        tasks.append(
            {
                "ecg":
                    ecg_key,

                "lead":
                    lead,

                "original_box":
                    [
                        x0,
                        y0,
                        x1,
                        y1,
                    ],

                "original_crop_width":
                    x1 - x0,

                "original_crop_height":
                    y1 - y0,

                "display_width":
                    crop.width,

                "display_height":
                    crop.height,

                "image":
                    pil_to_data_uri(
                        crop
                    ),
            }
        )


print(
    "Annotation tasks:",
    len(tasks)
)


# ======================================================================================
# 5. HTML / JAVASCRIPT APP
# ======================================================================================

tasks_json = json.dumps(
    tasks
)


html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>ECG waveform anchor annotation</title>

<style>

body {
    font-family:
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        Arial,
        sans-serif;

    margin: 0;
    background: #f4f4f4;
    color: #222;
}

header {
    position: sticky;
    top: 0;
    z-index: 10;
    background: white;
    border-bottom: 1px solid #ccc;
    padding: 12px 18px;
}

h1 {
    font-size: 21px;
    margin: 0 0 5px 0;
}

#progressText {
    font-weight: 600;
}

main {
    max-width: 1500px;
    margin: auto;
    padding: 18px;
}

.instructions {
    background: white;
    border-radius: 10px;
    padding: 14px 18px;
    margin-bottom: 16px;
    line-height: 1.5;
}

#title {
    font-size: 25px;
    font-weight: 700;
    margin: 8px 0 12px 0;
}

#canvasWrapper {
    width: 100%;
    overflow-x: auto;
    background: white;
    border-radius: 10px;
    padding: 10px;
    box-sizing: border-box;
}

canvas {
    display: block;
    max-width: none;
    cursor: crosshair;
    border: 1px solid #aaa;
}

.controls {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 14px;
}

button {
    border: 1px solid #999;
    border-radius: 7px;
    background: white;
    padding: 9px 14px;
    font-size: 14px;
    cursor: pointer;
}

button:hover {
    background: #eee;
}

button.primary {
    background: #222;
    color: white;
    border-color: #222;
}

button.keep {
    background: #e7f5e7;
}

button.corrected {
    background: #e7efff;
}

button.warning {
    background: #fff1d6;
}

#status {
    margin-top: 12px;
    padding: 9px 12px;
    border-radius: 7px;
    background: #eee;
}

#pointCount {
    margin-left: 10px;
    font-weight: 600;
}

kbd {
    border: 1px solid #bbb;
    border-radius: 4px;
    padding: 1px 5px;
    background: #eee;
}

</style>

</head>


<body>


<header>

<h1>ECG anchor annotation</h1>

<div id="progressText"></div>

</header>


<main>


<div class="instructions">

<strong>How to annotate each lead:</strong>

<br>

Click along the <strong>actual black ECG waveform</strong> from
<strong>left → right</strong>.

For each beat, include the important morphology:
baseline → QRS peak/trough → ST/T → next beat.

For large VT/LBBB/paced complexes, use more points around the QRS.

Aim for approximately <strong>15–30 points</strong> when correcting a lead.

<br><br>

A yellow line joins your manual anchor points.

The exact pixel-level trace will be reconstructed later in Python.

<br><br>

If the automatic V4 trace was already acceptable, choose
<strong>Keep V4</strong> rather than inventing a correction.

</div>


<div id="title"></div>


<div id="canvasWrapper">

<canvas id="canvas"></canvas>

</div>


<div class="controls">

<button onclick="previousTask()">← Previous</button>

<button onclick="undoPoint()">Undo point</button>

<button onclick="clearPoints()">Clear</button>

<button class="corrected"
        onclick="markCorrected()">
Save as corrected
</button>

<button class="keep"
        onclick="markKeep()">
Keep V4
</button>

<button onclick="nextTask()">Next →</button>

<button class="primary"
        onclick="exportJSON()">
Export JSON
</button>

<span id="pointCount"></span>

</div>


<div id="status"></div>


</main>


<script>


const tasks = __TASKS__;


const STORAGE_KEY =
    "ecg_anchor_annotations_v1";


let annotations = {};


try {

    const saved =
        localStorage.getItem(
            STORAGE_KEY
        );

    if (saved) {

        annotations =
            JSON.parse(
                saved
            );

    }

} catch (err) {

    console.log(
        err
    );

}


let currentIndex = 0;


const canvas =
    document.getElementById(
        "canvas"
    );


const ctx =
    canvas.getContext(
        "2d"
    );


const image =
    new Image();


function taskKey(task) {

    return (
        task.ecg
        +
        "::"
        +
        task.lead
    );

}


function getAnnotation(task) {

    const key =
        taskKey(task);

    if (!annotations[key]) {

        annotations[key] = {

            ecg:
                task.ecg,

            lead:
                task.lead,

            status:
                "pending",

            points:
                [],

            original_box:
                task.original_box,

            original_crop_width:
                task.original_crop_width,

            original_crop_height:
                task.original_crop_height,

            display_width:
                task.display_width,

            display_height:
                task.display_height

        };

    }

    return annotations[key];

}


function saveLocal() {

    localStorage.setItem(
        STORAGE_KEY,
        JSON.stringify(
            annotations
        )
    );

}


function renderTask() {

    const task =
        tasks[currentIndex];


    const ann =
        getAnnotation(
            task
        );


    document.getElementById(
        "title"
    ).innerText =
        task.ecg
        +
        " — "
        +
        task.lead;


    image.onload = function() {

        canvas.width =
            image.naturalWidth;

        canvas.height =
            image.naturalHeight;


        draw();

    };


    image.src =
        task.image;


    updateStatus();
    updateProgress();

}


function draw() {

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );


    ctx.drawImage(
        image,
        0,
        0,
        canvas.width,
        canvas.height
    );


    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    const points =
        ann.points;


    if (points.length === 0) {

        return;

    }


    // line
    ctx.strokeStyle =
        "#ffd400";

    ctx.lineWidth = 3;

    ctx.beginPath();


    points.forEach(
        (p, i) => {

            if (i === 0) {

                ctx.moveTo(
                    p.x,
                    p.y
                );

            } else {

                ctx.lineTo(
                    p.x,
                    p.y
                );

            }

        }
    );


    ctx.stroke();


    // points
    points.forEach(
        (p, i) => {

            ctx.beginPath();

            ctx.arc(
                p.x,
                p.y,
                5,
                0,
                2 * Math.PI
            );


            ctx.fillStyle =
                "#ff2b2b";

            ctx.fill();


            ctx.strokeStyle =
                "white";

            ctx.lineWidth = 1;

            ctx.stroke();


            ctx.font =
                "11px Arial";

            ctx.fillStyle =
                "black";

            ctx.fillText(
                String(i + 1),
                p.x + 6,
                p.y - 5
            );

        }
    );

}


canvas.addEventListener(
    "click",
    function(event) {

        const rect =
            canvas.getBoundingClientRect();


        const scaleX =
            canvas.width
            /
            rect.width;


        const scaleY =
            canvas.height
            /
            rect.height;


        const x =
            (
                event.clientX
                -
                rect.left
            )
            *
            scaleX;


        const y =
            (
                event.clientY
                -
                rect.top
            )
            *
            scaleY;


        const task =
            tasks[currentIndex];


        const ann =
            getAnnotation(
                task
            );


        // Enforce broadly left-to-right annotation.
        if (
            ann.points.length > 0
            &&
            x
            <
            ann.points[
                ann.points.length - 1
            ].x
        ) {

            alert(
                "Please place anchors from LEFT to RIGHT.\n\n"
                +
                "Use Undo if you need to go backwards."
            );

            return;

        }


        ann.points.push(
            {
                x: x,
                y: y
            }
        );


        ann.status =
            "editing";


        saveLocal();
        draw();
        updateStatus();

    }
);


function undoPoint() {

    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    if (
        ann.points.length
        >
        0
    ) {

        ann.points.pop();

        ann.status =
            "editing";

        saveLocal();
        draw();
        updateStatus();

    }

}


function clearPoints() {

    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    if (
        confirm(
            "Clear all anchor points for this lead?"
        )
    ) {

        ann.points = [];

        ann.status =
            "pending";

        saveLocal();
        draw();
        updateStatus();

    }

}


function markCorrected() {

    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    if (
        ann.points.length
        <
        5
    ) {

        alert(
            "Please place at least 5 anchor points "
            +
            "before marking this lead corrected."
        );

        return;

    }


    ann.status =
        "corrected";


    saveLocal();
    updateStatus();
    updateProgress();


    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        nextTask();

    }

}


function markKeep() {

    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    ann.status =
        "keep_v4";


    ann.points = [];


    saveLocal();
    draw();
    updateStatus();
    updateProgress();


    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        nextTask();

    }

}


function nextTask() {

    if (
        currentIndex
        <
        tasks.length - 1
    ) {

        currentIndex += 1;

        renderTask();

    }

}


function previousTask() {

    if (
        currentIndex
        >
        0
    ) {

        currentIndex -= 1;

        renderTask();

    }

}


function updateStatus() {

    const task =
        tasks[currentIndex];

    const ann =
        getAnnotation(
            task
        );


    const element =
        document.getElementById(
            "status"
        );


    let text =
        "Status: "
        +
        ann.status;


    if (
        ann.status
        ===
        "corrected"
    ) {

        element.style.background =
            "#e7efff";

    } else if (
        ann.status
        ===
        "keep_v4"
    ) {

        element.style.background =
            "#e7f5e7";

    } else {

        element.style.background =
            "#eeeeee";

    }


    element.innerText =
        text;


    document.getElementById(
        "pointCount"
    ).innerText =
        ann.points.length
        +
        " anchors";

}


function updateProgress() {

    let completed = 0;

    let corrected = 0;

    let kept = 0;


    tasks.forEach(
        task => {

            const ann =
                getAnnotation(
                    task
                );


            if (
                ann.status
                ===
                "corrected"
            ) {

                completed += 1;
                corrected += 1;

            }


            if (
                ann.status
                ===
                "keep_v4"
            ) {

                completed += 1;
                kept += 1;

            }

        }
    );


    document.getElementById(
        "progressText"
    ).innerText =
        "Lead "
        +
        (currentIndex + 1)
        +
        " / "
        +
        tasks.length
        +
        " | completed "
        +
        completed
        +
        " / "
        +
        tasks.length
        +
        " | corrected "
        +
        corrected
        +
        " | kept V4 "
        +
        kept;

}


function exportJSON() {

    saveLocal();


    const result = {

        format:
            "ECG_anchor_annotations_v1",

        task_count:
            tasks.length,

        annotations:
            annotations

    };


    const blob =
        new Blob(
            [
                JSON.stringify(
                    result,
                    null,
                    2
                )
            ],
            {
                type:
                    "application/json"
            }
        );


    const url =
        URL.createObjectURL(
            blob
        );


    const a =
        document.createElement(
            "a"
        );


    a.href =
        url;


    a.download =
        "ecg_anchor_annotations.json";


    document.body.appendChild(
        a
    );


    a.click();


    document.body.removeChild(
        a
    );


    URL.revokeObjectURL(
        url
    );

}


document.addEventListener(
    "keydown",
    function(event) {

        // Don't interfere with browser shortcuts
        if (
            event.ctrlKey
            ||
            event.metaKey
        ) {
            return;
        }


        const key =
            event.key.toLowerCase();


        if (
            key
            ===
            "u"
        ) {

            undoPoint();

        }


        if (
            key
            ===
            "c"
        ) {

            clearPoints();

        }


        if (
            key
            ===
            "n"
        ) {

            nextTask();

        }


        if (
            key
            ===
            "p"
        ) {

            previousTask();

        }


        if (
            key
            ===
            "k"
        ) {

            markKeep();

        }


        if (
            key
            ===
            "s"
        ) {

            markCorrected();

        }


        if (
            key
            ===
            "e"
        ) {

            exportJSON();

        }

    }
);


renderTask();


</script>


</body>

</html>
"""


html = html.replace(
    "__TASKS__",
    tasks_json
)


with open(
    OUTPUT_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html
    )


print("\n" + "=" * 100)
print("ANNOTATION TOOL CREATED")
print("=" * 100)

print(
    "\nFile:",
    OUTPUT_HTML.resolve()
)

print(
    "\nNumber of leads requiring review:",
    len(tasks)
)

print(
    """
Open the HTML file in Chrome or Safari.

For each lead:

  • Click anchors LEFT -> RIGHT.
  • Aim for ~15–30 useful points.
  • Include QRS peaks/troughs.
  • Use more points for VT/LBBB/paced complexes.
  • Click "Save as corrected" when satisfied.
  • Or click "Keep V4" if that lead was actually acceptable.

Progress is automatically saved in your browser.

When finished:

    Click "Export JSON"

The browser will download:

    ecg_anchor_annotations.json

Move that JSON file into the same project folder as this notebook.

Then tell me when you have it.
"""
)


# Try opening it automatically
try:

    webbrowser.open(
        OUTPUT_HTML.resolve().as_uri()
    )

except Exception:

    pass